## Vectores de soporte

In [1]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

def train_support_vector_machine(X, y): 
        
    C = [.001, .01, .1, 1, 10]
    kernels = ['linear', 'poly', 'rbf', 'sigmoid']
    
    rows = []
    for k in kernels:
        for c in C: 
            
            accs_train, accs_val, f1s, recalls = [], [], [], []
            
            kf = KFold(n_splits=5)
            for train_index, val_index in kf.split(X): 
                xtrain, xval = X[train_index], X[val_index]
                ytrain, yval = y[train_index], y[val_index]
                
                scaler = StandardScaler() 
                xtrain, xval = scaler.fit_transform(xtrain), scaler.transform(xval)
                
                model = SVC(C=c, kernel=k)
                model.fit(X=xtrain, y=ytrain)
                
                predictions_train = model.predict(xtrain)
                predictions_val = model.predict(xval)
                
                accs_train.append(accuracy_score(ytrain, predictions_train))
                accs_val.append(accuracy_score(yval, predictions_val))
                f1s.append(f1_score(yval, predictions_val))
                recalls.append(recall_score(yval, predictions_val))
                
            rows.append({    
                'kernel': k,
                'regularizacion C': c,
                'media acc train': np.mean(accs_train),
                'media acc val': np.mean(accs_val),
                'IC acc val': np.std(accs_val),
                'media f1': np.mean(f1s), 
                'IC f1': np.std(f1s), 
                'media recalls': np.mean(recalls), 
                'IC recalls': np.std(recalls)
            }) 
        print(f"----- finaliza para kernel = {k} ------")

    results = pd.DataFrame(rows)
    return results.sort_values(by='media f1', ascending=False)

Para los datos sin filtro de variables

In [2]:
from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() 

train_support_vector_machine(X_train, y_train)

----- finaliza para kernel = linear ------
----- finaliza para kernel = poly ------
----- finaliza para kernel = rbf ------
----- finaliza para kernel = sigmoid ------


,kernel,regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recalls,IC recalls
14,rbf,10.000,0.990344,0.970250,0.001458,0.970336,0.001124,0.972966,0.005025
9,poly,10.000,0.982937,0.960250,0.002114,0.959994,0.002379,0.954445,0.005401
13,rbf,1.000,0.970813,0.958750,0.003729,0.958910,0.003376,0.962218,0.004355
8,poly,1.000,0.961219,0.949625,0.002640,0.949092,0.002719,0.939479,0.004360
4,linear,10.000,0.948813,0.944875,0.002107,0.945133,0.001868,0.949707,0.004107
3,linear,1.000,0.947969,0.944375,0.001976,0.944681,0.001558,0.949958,0.004068
2,linear,0.100,0.945406,0.941250,0.002850,0.941737,0.002926,0.949949,0.004931
1,linear,0.010,0.939000,0.936750,0.002834,0.937437,0.003114,0.948200,0.005051
12,rbf,0.100,0.939625,0.932375,0.004246,0.933783,0.004470,0.954195,0.005478
0,linear,0.001,0.933938,0.932875,0.007155,0.933222,0.007659,0.939158,0.009083


Para los datos filtrados   

In [3]:
from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned()

train_support_vector_machine(X_train, y_train)

----- finaliza para kernel = linear ------
----- finaliza para kernel = poly ------
----- finaliza para kernel = rbf ------
----- finaliza para kernel = sigmoid ------


,kernel,regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recalls,IC recalls
14,rbf,10.000,0.990563,0.970125,0.007810,0.970183,0.007776,0.973894,0.007453
13,rbf,1.000,0.972125,0.961125,0.008220,0.961220,0.008056,0.965125,0.006051
9,poly,10.000,0.983437,0.958875,0.006305,0.958627,0.006383,0.955342,0.007996
8,poly,1.000,0.964312,0.953000,0.008134,0.952443,0.008394,0.944294,0.008362
4,linear,10.000,0.945312,0.943125,0.009626,0.943532,0.009227,0.951639,0.009148
3,linear,1.000,0.945063,0.942125,0.008835,0.942518,0.008365,0.950125,0.006197
2,linear,0.100,0.942562,0.940250,0.007812,0.940610,0.007662,0.948344,0.008415
1,linear,0.010,0.936312,0.935250,0.007661,0.935930,0.007294,0.947601,0.005721
12,rbf,0.100,0.936281,0.931375,0.008956,0.932817,0.008411,0.954405,0.003824
0,linear,0.001,0.931656,0.929625,0.009376,0.929721,0.009301,0.933054,0.008267


los dos mejores modelos tienen los parametros (kernel, regularizacion C)  igual a (rbf, 10) y (rbf, 1) respectivamente

In [10]:
from libs.data_configuration import get_filtered_data_partitioned
from libs.linear_reduction import train_models
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


xtrain, xtest, ytrain, ytest = get_filtered_data_partitioned() 

print(f"--- Iniciamos con {xtrain.shape[1]} features ---")
models = [
    SVC(kernel='rbf', C=10),
    SVC(kernel='rbf', C=1)
]

scaler = StandardScaler() 
xtrain, xtest = scaler.fit_transform(xtrain), scaler.transform(xtest)

pca = PCA(n_components=.95) # conservar el 95% de la variabilidad de los datos 
xtrain, xtest = pca.fit_transform(xtrain), pca.transform(xtest)

print(f"--- Terminamos con {xtest.shape[1]} features ---")

train_models(xtrain, xtest, ytrain, ytest, models, ['kernel', 'C'])

--- Iniciamos con 41 features ---
--- Terminamos con 32 features ---


,kernel,C,acc train,acc test,f1_score,recall
0,rbf,10,0.983375,0.9605,0.960242,0.963636
1,rbf,1,0.967500,0.9535,0.953290,0.958586
